# 06 - Feature Engineering, Round 2

Continues Phase 2 after `05_feature_engineering.ipynb` confirmed income
digits + frequency encoding as the best feature set so far (OOF
0.944174, LB 0.94425, rank 840). This notebook tests four more ideas on
top of that feature set - categorical N-grams (the open backlog item
from the public-notebook review), affordability/infrastructure ratios,
and quantile-binned income. Each is additive and compared against the
same 0.944174 baseline.

In [ ]:
import pandas as pd
import numpy as np
import time
from itertools import combinations

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import KBinsDiscretizer
from xgboost import XGBClassifier

train = pd.read_csv("../Data/train.csv")
test = pd.read_csv("../Data/test.csv")
TARGET = "Will_Buy_EV"
y = train[TARGET].map({"No": 0, "Yes": 1})
base_cat_cols = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"]

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
folds = list(skf.split(train, y))

def run_xgb_cv(X, Xt, tag=""):
    oof_pred = np.zeros(len(X))
    test_preds = np.zeros((N_SPLITS, len(Xt)))
    fold_aucs = []
    t0 = time.time()
    for fold, (tr_idx, va_idx) in enumerate(folds):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        model = XGBClassifier(
            n_estimators=3000, learning_rate=0.05, max_depth=6,
            tree_method="hist", device="cuda", enable_categorical=True,
            eval_metric="auc", early_stopping_rounds=50, random_state=42,
        )
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        va_pred = model.predict_proba(X_va)[:, 1]
        oof_pred[va_idx] = va_pred
        fold_aucs.append(roc_auc_score(y_va, va_pred))
        test_preds[fold] = model.predict_proba(Xt)[:, 1]
    oof_auc = roc_auc_score(y, oof_pred)
    print(f"[{tag}] mean_fold_auc={np.mean(fold_aucs):.6f} std={np.std(fold_aucs):.6f} OOF={oof_auc:.6f} time={time.time()-t0:.0f}s")
    return oof_auc, fold_aucs, oof_pred, test_preds.mean(axis=0)

print("ready")

ready


## Rebuild the current-best feature set

Income digit decomposition + frequency encoding of income and commute,
from Experiment 005. This is the baseline every idea below is compared
against.

In [ ]:
def add_income_digits(df):
    v = df["Annual_Income_USD"].astype(int)
    return pd.DataFrame({f"inc_d{i}": ((v // (10**i)) % 10).astype("category") for i in range(6)}, index=df.index)

def add_freq_features(df_tr, df_te, col):
    freq = df_tr[col].value_counts()
    return df_tr[col].map(freq).rename(f"{col}_freq"), df_te[col].map(freq).fillna(0).rename(f"{col}_freq")

X0 = train.drop(columns=[TARGET, "id"]).copy()
Xt0 = test.drop(columns=["id"]).copy()
for c in base_cat_cols:
    X0[c] = X0[c].astype("category")
    Xt0[c] = pd.Categorical(Xt0[c], categories=X0[c].cat.categories)

digits_tr = add_income_digits(train)
digits_te = add_income_digits(test)
inc_freq_tr, inc_freq_te = add_freq_features(train, test, "Annual_Income_USD")
com_freq_tr, com_freq_te = add_freq_features(train, test, "Daily_Commute_km")

X3 = pd.concat([X0, digits_tr, inc_freq_tr, com_freq_tr], axis=1)
Xt3 = pd.concat([Xt0, digits_te, inc_freq_te, com_freq_te], axis=1)

res_best = run_xgb_cv(X3, Xt3, "current best (income digits + freq)")

[current best (income digits + freq)] mean_fold_auc=0.944182 std=0.000555 OOF=0.944174 time=21s


Reproduces Experiment 005 exactly - confirmed via Kaggle too, this
feature set scored **public LB 0.94425, rank 840**.

## Idea 1: categorical pair frequency ("N-grams")

The open backlog item from the public-notebook review
(`evehicle-stacked-lgbm-catb-xgb-hgbc-baseline.ipynb` uses categorical
N-grams). Simplest version: frequency-encode every pairwise combination
of the 6 base categoricals (15 pairs), e.g. how often
`"Urban_SUV"` (City_Type + Current_Car_Type) appears in training data.

In [ ]:
def add_cat_pair_freq(df_tr, df_te, cols):
    tr_out, te_out = {}, {}
    for c1, c2 in combinations(cols, 2):
        combo_tr = df_tr[c1].astype(str) + "_" + df_tr[c2].astype(str)
        combo_te = df_te[c1].astype(str) + "_" + df_te[c2].astype(str)
        freq = combo_tr.value_counts()
        name = f"{c1}_{c2}_freq"
        tr_out[name] = combo_tr.map(freq)
        te_out[name] = combo_te.map(freq).fillna(0)
    return pd.DataFrame(tr_out, index=df_tr.index), pd.DataFrame(te_out, index=df_te.index)

pair_tr, pair_te = add_cat_pair_freq(train, test, base_cat_cols)

X4 = pd.concat([X3, pair_tr], axis=1)
Xt4 = pd.concat([Xt3, pair_te], axis=1)
res_pairs = run_xgb_cv(X4, Xt4, "+ categorical pair freq (N-grams)")

[+ categorical pair freq (N-grams)] mean_fold_auc=0.944078 std=0.000638 OOF=0.944074 time=27s


**-0.0001 vs. current best: no help, dropped.** 15 new columns from 6
already-modest-cardinality categoricals mostly add noise rather than new
signal - consistent with the earlier finding that target encoding and
the `City_Type` deviation feature also didn't help (Experiment 005):
XGBoost's native categorical splits already extract what these
categoricals have to offer on their own.

## Idea 2: affordability and infrastructure ratios

Domain-motivated ratios: income relative to age/cars owned (affordability
signals), commute relative to income, and charging-station totals/ratios
(infrastructure access, a different angle on the same idea that failed
as a `City_Type` deviation feature in Experiment 005).

In [ ]:
def add_ratio_features(df):
    out = {}
    out["income_per_age"] = df["Annual_Income_USD"] / (df["Age"] + 1)
    out["income_per_car"] = df["Annual_Income_USD"] / (df["Number_of_Cars_Owned"] + 1)
    out["commute_per_income"] = df["Daily_Commute_km"] / (df["Annual_Income_USD"] + 1)
    out["charging_total"] = df["Charging_Stations_Near_Home"] + df["Charging_Stations_Near_Work"]
    out["charging_home_ratio"] = df["Charging_Stations_Near_Home"] / (out["charging_total"] + 1)
    return pd.DataFrame(out, index=df.index)

ratio_tr = add_ratio_features(train)
ratio_te = add_ratio_features(test)

X5 = pd.concat([X3, ratio_tr], axis=1)
Xt5 = pd.concat([Xt3, ratio_te], axis=1)
res_ratios = run_xgb_cv(X5, Xt5, "+ affordability/infra ratios")

[+ affordability/infra ratios] mean_fold_auc=0.943989 std=0.000616 OOF=0.943986 time=26s


**-0.000188 vs. current best: no help, dropped.** A tree-based model
can already form these ratios implicitly via splits on the component
columns (which are all still present) - explicit ratio columns don't
add information XGBoost couldn't already reach, same pattern as Idea 1
and the Experiment 005 negative results.

## Idea 3: income quantile binning

A different mechanism from digit decomposition - bins income into 50
quantile-based groups (`KBinsDiscretizer`, ordinal-encoded), testing
whether a smoother, order-preserving income representation adds
anything the digit features (which are value-artifact-based, not
order-based) miss.

In [ ]:
kb = KBinsDiscretizer(n_bins=50, encode="ordinal", strategy="quantile")
income_bin_tr = kb.fit_transform(train[["Annual_Income_USD"]]).astype(int).ravel()
income_bin_te = kb.transform(test[["Annual_Income_USD"]]).astype(int).ravel()
bin_tr = pd.DataFrame({"income_qbin": pd.Categorical(income_bin_tr)}, index=train.index)
bin_te = pd.DataFrame({"income_qbin": pd.Categorical(income_bin_te, categories=bin_tr["income_qbin"].cat.categories)}, index=test.index)

X6 = pd.concat([X3, bin_tr], axis=1)
Xt6 = pd.concat([Xt3, bin_te], axis=1)
res_qbin = run_xgb_cv(X6, Xt6, "+ income quantile bins (50)")

[+ income quantile bins (50)] mean_fold_auc=0.943483 std=0.000556 OOF=0.943471 time=20s


**-0.000703 vs. current best: clearly no help, dropped** - the worst
of the three ideas tried this round. Quantile bins likely *compete* with
the digit features rather than complementing them: both encode income,
but the digit features carry the specific artifact signal the raw
generation process left behind, while quantile bins discard that in
favor of a coarser, order-based grouping. Adding a weaker, redundant
income representation alongside a stronger one seems to have made it
marginally harder for the trees to use the good one cleanly.

## Summary - Round 2

| Idea | OOF AUC | Δ vs. current best (0.944174) | Kept? |
|---|---|---|---|
| Current best (Experiment 005) | 0.944174 | - | - |
| + categorical pair frequency ("N-grams") | 0.944074 | -0.000100 | No |
| + affordability/infrastructure ratios | 0.943986 | -0.000188 | No |
| + income quantile bins (50) | 0.943471 | -0.000703 | No |

**No improvement found this round - all three ideas tested worse than
the existing feature set.** This is a legitimate, useful result: it
suggests the current feature set (income digits + frequency encoding)
is at or near a local plateau for straightforward additive feature
engineering on top of XGBoost's native categorical handling. The
consistent pattern across both this round and Experiment 005 - manual
interaction/ratio/encoding features not helping once XGBoost already
has the raw components to split on - suggests further gains are more
likely to come from a genuinely different mechanism (Optuna
hyperparameter tuning, ensembling/stacking across model families on the
*current* best feature set, or the entity-embedding DNN for structural
diversity) than from more manual feature construction of this kind.

**No new submission from this round** - none of the tested features
beat the existing best (`xgboost_5fold_income_digits_freq.csv`, LB
0.94425, rank 840), so there's nothing better to submit.